# Modeling with SigLIP embeddings (neutral dataset)

This notebook evaluates whether warmth and competence can be predicted from SigLIP embeddings extracted from the neutral-imputed dataset. In contrast to the strict dataset, this version includes a larger number of artworks but uses noisier targets due to neutral-value imputation.

The goal is to assess the trade-off between dataset size and label quality, and to determine whether broader coverage improves or weakens predictive performance.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

## 1. Loading the neutral dataset with SigLIP embeddings

The input for this notebook is the neutral-imputed dataset merged with SigLIP embeddings. Each row contains the original artwork metadata, the target variables, and the embedding dimensions (`emb_*`) that will be used as predictors.

In [2]:
DATA_PATH = "../Embeddings/siglip/final_dataset_neutral_with_siglip.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head(2)

Dataset shape: (6105, 791)


,cat_no,titulo,autor,escuela_obra,tipo_objeto,datacion,tema,is_religious,is_fauna,century,...,emb_758,emb_759,emb_760,emb_761,emb_762,emb_763,emb_764,emb_765,emb_766,emb_767
0,P000002,El juicio de Paris,"Albani, Francesco",Italiana,Pintura,1650 - 1660,NaN,0,1,17th c.,...,-0.356265,3.988893,-0.038276,-0.140301,0.248423,-0.136362,-0.621971,-0.149800,0.343511,0.144266
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",Italiana,Pintura,1584,NaN,1,1,16th c.,...,0.258788,4.340028,-0.456233,0.113226,0.123200,-0.487182,-0.321165,-0.323432,0.417856,0.013111


## 2. Preparing features and targets

The embedding vectors (`emb_*`) are used as input features, while warmth and competence scores are used as target variables. These targets include imputed values in the neutral dataset.

In [3]:
# Select embedding columns
embedding_cols = [col for col in df.columns if col.startswith("emb_")]

X = df[embedding_cols].values

# Targets
y_warmth = df["dirmean_Warmth"].values
y_competence = df["dirmean_Competence"].values

print("Feature matrix shape:", X.shape)
print("Number of embedding features:", len(embedding_cols))

Feature matrix shape: (6105, 768)
Number of embedding features: 768


## 3. Train-test split

The dataset is split into training and test sets using a fixed random seed to ensure reproducibility and comparability with previous experiments.

In [4]:
X_train, X_test, y_w_train, y_w_test = train_test_split(
    X, y_warmth, test_size=0.2, random_state=42
)

_, _, y_c_train, y_c_test = train_test_split(
    X, y_competence, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 4884
Test size: 1221


## 4. Mean baseline

Before training models, a simple baseline is computed by predicting the mean value of the training set. This provides a reference point to evaluate whether the models are actually learning meaningful patterns.

In [5]:
def evaluate(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f"{name} R²: {r2:.4f}")
    print(f"{name} RMSE: {rmse:.4f}")
    print("-" * 30)

# Baseline predictions
y_w_baseline = np.repeat(y_w_train.mean(), len(y_w_test))
y_c_baseline = np.repeat(y_c_train.mean(), len(y_c_test))

evaluate(y_w_test, y_w_baseline, "Warmth baseline")
evaluate(y_c_test, y_c_baseline, "Competence baseline")

Warmth baseline R²: -0.0006
Warmth baseline RMSE: 0.5381
------------------------------
Competence baseline R²: -0.0011
Competence baseline RMSE: 0.3184
------------------------------


## 5. Linear model: Ridge regression

A Ridge regression model is used to test whether warmth and competence can be recovered through a linear mapping from the embedding space. This serves as a baseline for understanding whether the relationship between visual features and the targets is approximately linear.

In [6]:
# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train models
ridge_w = Ridge(alpha=1.0)
ridge_c = Ridge(alpha=1.0)

ridge_w.fit(X_train_scaled, y_w_train)
ridge_c.fit(X_train_scaled, y_c_train)

# Predictions
y_w_pred_ridge = ridge_w.predict(X_test_scaled)
y_c_pred_ridge = ridge_c.predict(X_test_scaled)

# Evaluate
evaluate(y_w_test, y_w_pred_ridge, "Warmth Ridge")
evaluate(y_c_test, y_c_pred_ridge, "Competence Ridge")

Warmth Ridge R²: -0.0722
Warmth Ridge RMSE: 0.5570
------------------------------
Competence Ridge R²: -0.1136
Competence Ridge RMSE: 0.3358
------------------------------


## 6. Non-linear model: Random Forest

A Random Forest regressor is used to capture potential non-linear relationships between the embedding space and the target variables. This allows us to evaluate whether the information is present but not linearly separable.

In [7]:
rf_w = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_c = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Train
rf_w.fit(X_train, y_w_train)
rf_c.fit(X_train, y_c_train)

# Predict
y_w_pred_rf = rf_w.predict(X_test)
y_c_pred_rf = rf_c.predict(X_test)

# Evaluate
evaluate(y_w_test, y_w_pred_rf, "Warmth RF")
evaluate(y_c_test, y_c_pred_rf, "Competence RF")

Warmth RF R²: 0.0389
Warmth RF RMSE: 0.5273
------------------------------
Competence RF R²: -0.0211
Competence RF RMSE: 0.3215
------------------------------


## 7. Results summary

The table below compares the performance of the mean baseline, the linear Ridge model, and the Random Forest model for both warmth and competence.

In [8]:
results = pd.DataFrame([
    ["Warmth", "Baseline", r2_score(y_w_test, y_w_baseline), np.sqrt(mean_squared_error(y_w_test, y_w_baseline))],
    ["Warmth", "Ridge", r2_score(y_w_test, y_w_pred_ridge), np.sqrt(mean_squared_error(y_w_test, y_w_pred_ridge))],
    ["Warmth", "Random Forest", r2_score(y_w_test, y_w_pred_rf), np.sqrt(mean_squared_error(y_w_test, y_w_pred_rf))],
    
    ["Competence", "Baseline", r2_score(y_c_test, y_c_baseline), np.sqrt(mean_squared_error(y_c_test, y_c_baseline))],
    ["Competence", "Ridge", r2_score(y_c_test, y_c_pred_ridge), np.sqrt(mean_squared_error(y_c_test, y_c_pred_ridge))],
    ["Competence", "Random Forest", r2_score(y_c_test, y_c_pred_rf), np.sqrt(mean_squared_error(y_c_test, y_c_pred_rf))]
], columns=["Target", "Model", "R²", "RMSE"])

results

,Target,Model,R²,RMSE
0,Warmth,Baseline,-0.000552,0.538050
1,Warmth,Ridge,-0.072185,0.556978
2,Warmth,Random Forest,0.038877,0.527342
3,Competence,Baseline,-0.001064,0.318350
4,Competence,Ridge,-0.113564,0.335762
5,Competence,Random Forest,-0.021098,0.321520


## 8. Interpretation of modeling results (neutral dataset)

The results obtained on the neutral dataset reinforce several important insights regarding the relationship between visual embeddings and the SCM dimensions.

First, the baseline model establishes a strong reference point. Since both warmth and competence distributions are highly imbalanced, predicting the mean already yields near-zero R² values. This indicates that any useful model must outperform a very strong central tendency.

The Ridge regression model performs worse than the baseline for both dimensions. This suggests that the relationship between the visual embeddings and the target variables is not linearly separable. In other words, a simple linear mapping is insufficient to recover warmth and competence from the embedding space.

The Random Forest model shows a slight improvement for warmth (R² ≈ 0.04), indicating that some non-linear signal is present in the embeddings. However, this improvement remains modest, and the model still struggles to capture strong predictive patterns. For competence, the model fails to outperform the baseline, with negative R² values, confirming that competence remains particularly difficult to predict.

When comparing these results with those obtained on the strict dataset, a clear pattern emerges. Despite having fewer samples, the strict dataset achieved better performance, particularly for warmth prediction. In contrast, the neutral dataset, although larger, leads to weaker results.

This highlights a key trade-off in the data: increasing dataset size introduces noisier or less reliable labels, which ultimately degrades model performance. Therefore, label quality appears to be more important than dataset size in this setting.

Overall, these findings suggest that:
- Visual embeddings contain limited but non-linear information related to warmth  
- Competence is highly constrained by label distribution and exhibits weak learnability  
- The predictive signal is sensitive to data quality, with stricter filtering leading to better outcomes  

These results support the idea that while visual features encode some socially relevant information, recovering structured stereotype dimensions such as warmth and competence remains a challenging task.